# Build the BIH air-quality dataset

Merges the FHZ and RHZ sources into one hourly dataset. Fill in the Drive file
IDs in the next cell and this runs anywhere, with nothing to upload.

**Why this replaces the old merge.** `rhz_data.csv` was merged by column
*position*, but the source workbooks change column order between years inside
the same file. Banja Luka is `PM2,5 PM10 Temp ... CO NO NO2 NOx O3 SO2` in
2021-22 and `NO NO2 NOx SO2 AtPr ... PM10 Temp O3 CO PM2,5` in 2023-25, so the
labels ended up on the wrong data: in 2024 `no2_1h` actually held temperature,
`co_1h` held wind direction and `wind_speed` held atmospheric pressure.

Everything below maps columns **by name, per sheet**.

It also fixes the wind data. The FHZ wind export stacks two readings per
timestamp in one column - row 0 is direction (0-360) and row 1 is speed - so
every previous `wind_speed` value was an average of a bearing and a speed.
Here they are split, direction is averaged *circularly*, and RHZ km/h is
converted to m/s.

In [2]:
FILE_IDS = {
    "rhz_data/Banja Luka.xlsx":                        "12UEzOc5h3l4J9BvzhtGAxeavR_W2cj5j",
    "rhz_data/Bijeljina.xls":                          "1wQPi0LuC1I58P02-p8kG_0naKaiCK_Zx",
    "rhz_data/Brod.xlsx":                              "15vg3CuTRX9BjsgvTi8ZtLo-7PvX57PqS",
    "rhz_data/Doboj.xlsx":                             "1CF5p8hhmziDadzLT6np2wx8_7osNudF5",
    "rhz_data/Gacko.xls":                              "1dSZFdrA1ybgRk1qColntGu3a_Qjmrw-D",
    "rhz_data/Prijedor.xlsx":                          "1pf6QoOP8_-rcGaIiDTrb7xrOa2fGP_oT",
    "rhz_data/Trebinje.xlsx":                          "1PQ70oyJv57lVT4sqObO85I1mmRGK7LJ_",
    "rhz_data/Ugljevika.xlsx":                         "1VwHMJ9E7qQAH-ST6MhKvEEoOL_9ph9Gz",
    "fhz_data/fhz_data.csv":                           "1l2tEy4pfaO_ubYVFMsqPHgUn73UeT-DV",
    "fhz_data/wind/fhz_windspeed.csv":                 "1kU1DxMAV4x5cWNrJBrKK7tH9U5WsNqgd",
    "fhz_data/temperature_1h/fhz_temperature_1h.csv":  "1aGdVijSX4Tj1JI4pd5b9UbTrboj3D2aY",
}

# DRIVE_DATASET = "/content/drive/MyDrive/air_pollution_bih/dataset"  # if mounting
# LOCAL_DATASET = "/home/sehy/Downloads/air_pollution_bih/dataset"    # if running locally

import os, re, sys, subprocess
import numpy as np
import pandas as pd


def _gdown():
    """Import gdown, installing it only if something actually needs downloading."""
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    return gdown


def fetch_from_drive(file_ids, dest="dataset"):
    """Download each source file by its Drive ID, skipping ones already here."""
    gdown = None
    for rel, fid in file_ids.items():
        out = os.path.join(dest, rel)
        if os.path.exists(out) and os.path.getsize(out) > 0:
            print(f"have  {rel}")
            continue
        gdown = gdown or _gdown()
        os.makedirs(os.path.dirname(out), exist_ok=True)
        print(f"get   {rel}")
        gdown.download(id=fid, output=out, quiet=False)
        if not os.path.exists(out) or os.path.getsize(out) == 0:
            raise RuntimeError(
                f"download failed for {rel}. Check the ID and that the file is "
                f"shared as 'Anyone with the link'."
            )
    return dest


filled = {k: v for k, v in FILE_IDS.items() if v.strip()}
if filled:
    if len(filled) < len(FILE_IDS):
        missing = [k for k in FILE_IDS if not FILE_IDS[k].strip()]
        raise SystemExit(f"Some file IDs are still blank: {missing}")
    DATASET = fetch_from_drive(FILE_IDS)
elif os.path.isdir(LOCAL_DATASET):
    DATASET = LOCAL_DATASET
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATASET = DRIVE_DATASET

print("\ndataset dir:", DATASET, "| exists:", os.path.isdir(DATASET))

have  rhz_data/Banja Luka.xlsx
have  rhz_data/Bijeljina.xls
have  rhz_data/Brod.xlsx
get   rhz_data/Doboj.xlsx


Downloading...
From: https://drive.google.com/uc?id=1CF5p8hhmziDadzLT6np2wx8_7osNudF5
To: /content/dataset/rhz_data/Doboj.xlsx
100%|██████████| 2.77M/2.77M [00:00<00:00, 158MB/s]


get   rhz_data/Gacko.xls


Downloading...
From: https://drive.google.com/uc?id=1dSZFdrA1ybgRk1qColntGu3a_Qjmrw-D
To: /content/dataset/rhz_data/Gacko.xls
100%|██████████| 9.27M/9.27M [00:00<00:00, 14.4MB/s]


get   rhz_data/Prijedor.xlsx


...

get   fhz_data/wind/fhz_windspeed.csv


Downloading...
From: https://drive.google.com/uc?id=1kU1DxMAV4x5cWNrJBrKK7tH9U5WsNqgd
To: /content/dataset/fhz_data/wind/fhz_windspeed.csv
100%|██████████| 77.8M/77.8M [00:01<00:00, 68.6MB/s]


get   fhz_data/temperature_1h/fhz_temperature_1h.csv


Downloading...
From: https://drive.google.com/uc?id=1aGdVijSX4Tj1JI4pd5b9UbTrboj3D2aY
To: /content/dataset/fhz_data/temperature_1h/fhz_temperature_1h.csv
100%|██████████| 8.82M/8.82M [00:00<00:00, 22.6MB/s]


dataset dir: dataset | exists: True


## Parser - maps every source column by name

In [3]:
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- canonical names

# Bijeljina's workbook uses Cyrillic homoglyphs (С in "СО", О in "О3", Ѕ in "ЅО2")
# that look identical to Latin letters but break naive name matching.
CYRILLIC_TO_LATIN = str.maketrans({
    "А": "A", "В": "B", "С": "C", "Е": "E", "Н": "H", "І": "I", "Ј": "J",
    "К": "K", "М": "M", "О": "O", "Р": "P", "Ѕ": "S", "Т": "T", "У": "Y",
    "Х": "X", "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "х": "x",
})

SYNONYMS = {
    "pm1":         ["pm1"],
    "pm25":        ["pm25", "pm2.5", "pm2,5"],
    "pm4":         ["pm4"],
    "pm10":        ["pm10"],
    "so2":         ["so2"],
    "no":          ["no"],
    "no2":         ["no2"],
    "nox":         ["nox"],
    "o3":          ["o3"],
    "co":          ["co"],
    "co2":         ["co2"],
    "h2s":         ["h2s"],
    "benzene":     ["benz", "benzen"],
    "soot":        ["cad", "cadj"],
    "temperature": ["temp", "temzraka", "temparatura", "temperatura"],
    "humidity":    ["revl", "relvl", "vlagazraka", "vlaznostvazduha"],
    "wind_speed":  ["brvj", "brzinavjetra"],
    "wind_dir":    ["smvj", "smjervjetra"],
    "pressure":    ["atpr", "pritisak"],
    "solar":       ["sunzrac", "glsu", "suncszrace", "suncizrace"],
    "rain":        ["kolickise", "kolkise"],
}
LOOKUP = {alias: canon for canon, aliases in SYNONYMS.items() for alias in aliases}

# units that mean the column needs converting to our target unit
TARGET_UNITS = {"wind_speed": "m/s", "co": "mg/m3"}

POLLUTANTS = ["pm10", "pm25", "so2", "no2", "o3", "co"]
WEATHER = ["temperature", "humidity", "wind_speed", "wind_dir", "pressure"]


def norm(name: str) -> str:
    """Normalise a raw header cell to a comparable key."""
    s = str(name).translate(CYRILLIC_TO_LATIN).lower().strip()
    s = s.replace("µ", "u")
    s = re.sub(r"[\s\.\,\-_/]+", "", s)
    return s


def canonical(name: str):
    """Map a raw header cell to a canonical variable name, or None."""
    key = norm(name)
    if key in LOOKUP:
        return LOOKUP[key]
    # 'pm 2.5 ' -> 'pm25', 'PM25' -> 'pm25'
    key2 = key.replace(",", ".").replace(".", "")
    return LOOKUP.get(key2)


def convert_units(series: pd.Series, canon: str, unit: str) -> pd.Series:
    """Harmonise to m/s for wind and mg/m3 for CO."""
    u = norm(unit)
    if canon == "wind_speed" and u in ("kmh", "km/h", "kmph"):
        return series / 3.6
    if canon == "co" and u in ("ugm3", "ug/m3"):
        return series / 1000.0
    return series


# ---------------------------------------------------------------- rhz hourly workbooks

def parse_rhz_sheet(book, sheet: str) -> pd.DataFrame:
    """
    Parse one year-sheet of an rhz workbook.

    Layout: row0 = station group (sparse, forward-filled), row1 = variable name,
    row2 = unit, row3+ = data with col 0 = timestamp. Most sheets then repeat the
    whole variable block as daily means ('Dnevne srednje vrijednosti'); we cut at
    that marker, or at the first repeat of a (group, variable) pair.
    """
    raw = book.parse(sheet, header=None)
    groups = [str(v).strip() for v in raw.iloc[0]]
    names = [str(v).strip() for v in raw.iloc[1]]
    units = [str(v).strip() for v in raw.iloc[2]]

    # forward-fill the sparse station-group row
    cur = None
    filled = []
    for g in groups:
        if g not in ("nan", ""):
            cur = g
        filled.append(cur)

    keep, seen = [], set()
    for i in range(1, len(names)):
        nm = names[i]
        if nm in ("nan", "", "NaT") or "nevne" in nm.lower():
            break
        key = (filled[i], nm)
        if key in seen:            # start of the repeated daily-mean block
            break
        seen.add(key)
        canon = canonical(nm)
        if canon:
            keep.append((i, filled[i], canon, units[i]))

    body = raw.iloc[3:]
    ts = pd.to_datetime(body[0], dayfirst=True, errors="coerce")
    out = pd.DataFrame({"datetime": ts.values})

    # If a city has several station groups, the first group is the air-quality
    # station; a later met station (e.g. 'MS B Luka') only fills weather gaps.
    primary = keep[0][1] if keep else None
    for idx, grp, canon, unit in keep:
        col = pd.to_numeric(body[idx], errors="coerce").values
        col = convert_units(pd.Series(col), canon, unit)
        if canon not in out.columns:
            out[canon] = col.values
        elif grp != primary:
            out[canon] = out[canon].fillna(pd.Series(col.values, index=out.index))

    return out.dropna(subset=["datetime"])


def parse_rhz_workbook(path: str, city: str) -> pd.DataFrame:
    book = pd.ExcelFile(path)
    frames = []
    for sheet in book.sheet_names:
        try:
            df = parse_rhz_sheet(book, sheet)
        except Exception as exc:  # noqa: BLE001
            print(f"  ! {city} [{sheet}] failed: {exc}")
            continue
        if len(df):
            df["sheet"] = sheet
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    out["city"] = city
    out["station"] = city
    out["source"] = "rhz"
    return out


# ---------------------------------------------------------------- Bijeljina (daily)

def parse_bijeljina(path: str) -> pd.DataFrame:
    """Bijeljina is DAILY, not hourly, and uses Cyrillic homoglyph headers."""
    book = pd.ExcelFile(path)
    frames = []
    for sheet in book.sheet_names:
        raw = book.parse(sheet, header=None)
        names = [str(v).strip() for v in raw.iloc[1]]
        units = [str(v).strip() for v in raw.iloc[2]]
        body = raw.iloc[3:]
        ts = pd.to_datetime(body[0], dayfirst=True, errors="coerce")
        out = pd.DataFrame({"date": ts.values})
        for i in range(1, len(names)):
            canon = canonical(names[i])
            if canon and canon not in out.columns:
                col = pd.to_numeric(body[i], errors="coerce")
                out[canon] = convert_units(col, canon, units[i]).values
        out = out.dropna(subset=["date"])
        if len(out):
            frames.append(out)
    df = pd.concat(frames, ignore_index=True)
    df["city"] = df["station"] = "Bijeljina"
    df["source"] = "rhz"
    return df


# ---------------------------------------------------------------- fhz weather

def circular_mean_deg(values: pd.Series) -> float:
    """Mean of compass bearings; a plain mean of 350 and 10 gives 180, not 0."""
    v = values.dropna()
    if v.empty:
        return np.nan
    rad = np.deg2rad(v.to_numpy())
    ang = np.rad2deg(np.arctan2(np.sin(rad).mean(), np.cos(rad).mean()))
    return ang + 360 if ang < 0 else ang


def parse_fhz_wind(path: str) -> pd.DataFrame:
    """
    The wind export stacks two readings per timestamp in a single 'wind_speed'
    column: row 0 is direction (0-360, whole numbers), row 1 is speed (m/s).
    Verified: position 0 maxes at exactly 360.0 in all 9 cities and is 99.9%
    integers; position 1 has mean 1.8 m/s.
    Data is 10-minutely and is aggregated up to hourly.
    """
    w = pd.read_csv(path, low_memory=False)
    w["pos"] = w.groupby(["city", "year", "month", "day", "hour"]).cumcount()
    w = w[w.pos < 2]
    w["hh"] = w.hour.str.slice(0, 2).astype(int)
    w["datetime"] = pd.to_datetime(
        dict(year=w.year.astype(int), month=w.month.astype(int),
             day=w.day.astype(int), hour=w.hh), errors="coerce"
    )
    w = w.dropna(subset=["datetime"])

    spd = w[w.pos == 1].copy()
    dirn = w[w.pos == 0].copy()
    spd.loc[spd.wind_speed > 50, "wind_speed"] = np.nan   # Bihac has a few 282 m/s spikes
    dirn.loc[(dirn.wind_speed < 0) | (dirn.wind_speed > 360), "wind_speed"] = np.nan

    s = spd.groupby(["city", "datetime"]).wind_speed.mean().rename("wind_speed")
    d = dirn.groupby(["city", "datetime"]).wind_speed.apply(circular_mean_deg).rename("wind_dir")
    return pd.concat([s, d], axis=1).reset_index()


def parse_fhz_temp(path: str) -> pd.DataFrame:
    t = pd.read_csv(path, low_memory=False)
    t["hh"] = t.hour.str.slice(0, 2).astype(int)
    t["datetime"] = pd.to_datetime(
        dict(year=t.year.astype(int), month=t.month.astype(int),
             day=t.day.astype(int), hour=t.hh), errors="coerce"
    )
    t = t.dropna(subset=["datetime"])
    t["city"] = t.city.replace({"Sanski": "Sanski Most"})   # spelled differently across the two files
    return t.groupby(["city", "datetime"]).temperature.mean().reset_index()

## Build

In [4]:
RHZ_HOURLY = {
    "Banja Luka": "Banja Luka.xlsx",
    "Brod":       "Brod.xlsx",
    "Doboj":      "Doboj.xlsx",
    "Gacko":      "Gacko.xls",
    "Prijedor":   "Prijedor.xlsx",
    "Trebinje":   "Trebinje.xlsx",
    "Ugljevik":   "Ugljevika.xlsx",
}

SEASONS = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring",
           5: "spring", 6: "summer", 7: "summer", 8: "summer", 9: "autumn",
           10: "autumn", 11: "autumn"}

# ---- rhz hourly workbooks -------------------------------------------
frames = []
for city, fname in RHZ_HOURLY.items():
    df = parse_rhz_workbook(os.path.join(DATASET, "rhz_data", fname), city)
    got = [c for c in POLLUTANTS + WEATHER if c in df.columns]
    print(f"rhz {city:12s} rows={len(df):6d} vars={got}")
    frames.append(df)
rhz = pd.concat(frames, ignore_index=True)

# ---- fhz pollutants (already correctly mapped) + rebuilt weather ----
fhz = pd.read_csv(os.path.join(DATASET, "fhz_data", "fhz_data.csv"), low_memory=False)
fhz = fhz.rename(columns={c: c.replace("_1h", "") for c in fhz.columns})
fhz["datetime"] = pd.to_datetime(fhz["datetime"], format="mixed", errors="coerce")
fhz = fhz.dropna(subset=["datetime"])
fhz["datetime"] = fhz["datetime"].dt.round("h")
fhz = fhz.drop(columns=[c for c in ["wind_speed", "temperature", "season",
                                    "year", "month", "day", "hour"] if c in fhz.columns])
fhz["source"] = "fhz"

wind = parse_fhz_wind(os.path.join(DATASET, "fhz_data", "wind", "fhz_windspeed.csv"))
temp = parse_fhz_temp(os.path.join(DATASET, "fhz_data", "temperature_1h",
                                   "fhz_temperature_1h.csv"))
print(f"\nfhz wind rows={len(wind)}  temp rows={len(temp)}")
fhz = fhz.merge(wind, on=["city", "datetime"], how="left")
fhz = fhz.merge(temp, on=["city", "datetime"], how="left")

# ---- combine --------------------------------------------------------
bih = pd.concat([fhz, rhz], ignore_index=True)
for c in POLLUTANTS + WEATHER:
    if c not in bih.columns:
        bih[c] = np.nan
    bih[c] = pd.to_numeric(bih[c], errors="coerce")

for c in POLLUTANTS:
    bih.loc[bih[c] < 0, c] = np.nan
bih.loc[bih.pm25 > bih.pm10, ["pm25", "pm10"]] = np.nan

LIMITS = {"wind_dir": (0, 360), "wind_speed": (0, 60), "humidity": (0, 100),
          "pressure": (800, 1100), "temperature": (-40, 50)}
for c, (lo, hi) in LIMITS.items():
    n = int(((bih[c] < lo) | (bih[c] > hi)).sum())
    if n:
        print(f"  clipped {n} out-of-range {c}")
    bih.loc[(bih[c] < lo) | (bih[c] > hi), c] = np.nan

# stray stamps: a 2012 typo in Prijedor, and 2026-01-01 00:00 rows that are
# really the last hour of 2025
bih = bih[bih.datetime.dt.year.between(2021, 2025)]

# fhz emits two partial rows per station-hour (one carries pm10, its twin
# carries pm25). Of 61k such groups only ~20 disagree by >1%, so averaging the
# non-null values recombines them without losing information.
before = len(bih)
bih = (bih.groupby(["source", "city", "station", "datetime"], as_index=False)
          [POLLUTANTS + WEATHER].mean())
print(f"  merged {before - len(bih)} duplicate station-hour rows")

bih["year"] = bih.datetime.dt.year
bih["month"] = bih.datetime.dt.month
bih["day"] = bih.datetime.dt.day
bih["hour"] = bih.datetime.dt.hour
bih["season"] = bih.month.map(SEASONS)
bih = bih[["source", "city", "station", "datetime", "year", "month", "day",
           "hour", "season"] + POLLUTANTS + WEATHER]
bih = bih.sort_values(["city", "station", "datetime"]).reset_index(drop=True)
bih.to_csv(os.path.join(DATASET, "bih_hourly.csv"), index=False)

# ---- Bijeljina, kept separate because it is DAILY --------------------
bij = parse_bijeljina(os.path.join(DATASET, "rhz_data", "Bijeljina.xls"))
bij.to_csv(os.path.join(DATASET, "bih_daily_bijeljina.csv"), index=False)

print(f"\nbih_hourly.csv {bih.shape} cities={bih.city.nunique()} stations={bih.station.nunique()}")
print(f"bih_daily_bijeljina.csv {bij.shape}")

rhz Banja Luka   rows= 43824 vars=['pm10', 'pm25', 'so2', 'no2', 'o3', 'co', 'temperature', 'humidity', 'wind_speed', 'wind_dir', 'pressure']
rhz Brod         rows= 43824 vars=['pm10', 'pm25', 'so2', 'no2', 'o3', 'co', 'temperature', 'humidity', 'wind_speed', 'wind_dir', 'pressure']
rhz Doboj        rows= 35064 vars=['pm10', 'so2', 'no2', 'o3', 'co', 'humidity', 'wind_speed', 'wind_dir', 'pressure']
rhz Gacko        rows= 43824 vars=['pm10', 'so2', 'no2', 'temperature', 'humidity', 'wind_speed', 'wind_dir', 'pressure']
rhz Prijedor     rows= 43824 vars=['pm10', 'pm25', 'so2', 'no2', 'o3', 'co', 'temperature', 'humidity', 'wind_speed', 'wind_dir', 'pressure']
rhz Trebinje     rows= 43824 vars=['pm10', 'pm25', 'so2', 'no2', 'o3', 'co', 'temperature', 'humidity', 'wind_speed', 'wind_dir', 'pressure']
rhz Ugljevik     rows= 43824 vars=['pm10', 'so2', 'no2', 'temperature', 'humidity', 'wind_speed', 'wind_dir', 'pressure']

fhz wind rows=358867  temp rows=316183
  clipped 1 out-of-range wind

## Sanity check

In [5]:
print("duplicate station-hours:", int(bih.duplicated(["station", "datetime"]).sum()))
print()
print("plausibility of every rebuilt column:")
print(bih[POLLUTANTS + WEATHER].describe(percentiles=[.5]).T[
    ["count", "mean", "50%", "max"]].round(2).to_string())

duplicate station-hours: 0

plausibility of every rebuilt column:
                count    mean     50%      max
pm10         514556.0   33.01   22.00   749.86
pm25         371828.0   26.35   14.39   627.79
so2          619271.0   23.04   10.98  2121.40
no2          641400.0   16.88   11.36   272.50
o3           398615.0   51.18   43.00   720.70
co           385689.0    0.64    0.35    10.60
temperature  682761.0   12.42   12.10    41.60
humidity     213249.0   69.64   74.52   100.00
wind_speed   763630.0    1.62    1.30    33.00
wind_dir     754927.0  177.89  162.00   360.00
pressure     189831.0  977.50  995.70  1025.00


## Validation

Confirms the rebuilt columns actually match the source. This should print
`ALL MATCH` for every city and year, including across the year boundary where
the source layout changes.

In [6]:
# Re-read three workbooks straight from the sheet, by name, and confirm every
# column in bih_hourly matches its correctly-named source column. This is the
# test that caught the original bug: with positional parsing, Banja Luka 2024
# "no2" was actually temperature and "wind_speed" was atmospheric pressure.
MAP = {"pm10": "PM10", "pm25": "PM2,5", "so2": "SO2", "no2": "NO2", "o3": "O3",
       "co": "CO", "temperature": "Temp", "wind_speed": "BrVj", "wind_dir": "SmVj"}

for city, fname in [("Banja Luka", "Banja Luka.xlsx"), ("Brod", "Brod.xlsx"),
                    ("Ugljevik", "Ugljevika.xlsx")]:
    book = pd.ExcelFile(os.path.join(DATASET, "rhz_data", fname))
    for yr in ["2022", "2024"]:
        if yr not in book.sheet_names:
            continue
        raw = book.parse(yr, header=None)
        names = [str(v).strip() for v in raw.iloc[1]]
        units = [str(v).strip() for v in raw.iloc[2]]
        body = raw.iloc[3:]
        ts = pd.to_datetime(body[0], dayfirst=True, errors="coerce")
        sub = bih[(bih.city == city) & (bih.year == int(yr))].set_index("datetime")
        bad = []
        for canon, srcname in MAP.items():
            idxs = [i for i, n in enumerate(names) if n == srcname]
            if not idxs or canon not in sub:
                continue
            i = idxs[0]
            truth = pd.Series(pd.to_numeric(body[i], errors="coerce").values,
                              index=ts.values).dropna()
            if units[i].strip() == "km/h" and canon == "wind_speed":
                truth = truth / 3.6
            j = pd.DataFrame({"got": sub[canon]}).join(truth.rename("want"),
                                                       how="inner").dropna()
            if len(j) < 50:
                continue
            c = j.got.corr(j.want)
            if c < 0.999:
                bad.append(f"{canon}:{c:.3f}")
        print(f"{city:11s} {yr}  {'ALL MATCH' if not bad else 'MISMATCH ' + str(bad)}")

Banja Luka  2022  ALL MATCH
Banja Luka  2024  ALL MATCH
Brod        2022  ALL MATCH
Brod        2024  ALL MATCH
Ugljevik    2022  ALL MATCH
Ugljevik    2024  ALL MATCH


In [7]:
import os, zipfile
from google.colab import files

with zipfile.ZipFile("bih_dataset.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["bih_hourly.csv", "bih_daily_bijeljina.csv"]:
        p = os.path.join(DATASET, name)
        if os.path.exists(p):
            z.write(p, name)
            print(f"added {name}  {os.path.getsize(p)/1e6:.1f} MB")

print(f"\nzip: {os.path.getsize('bih_dataset.zip')/1e6:.1f} MB")
files.download("bih_dataset.zip")

added bih_hourly.csv  102.2 MB
added bih_daily_bijeljina.csv  0.2 MB

zip: 25.6 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>